In [19]:
import torch
import os

# Tối ưu hóa CUDA memory
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# Nếu chạy lại, clear GPU
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print("✅ GPU memory reset")

✅ GPU memory reset


In [20]:
pip install --upgrade huggingface_hub datasets

  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.13.0-py3-none-any.whl (660 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.44.2 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.13.0 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.13.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [22]:
# import os
# import glob
# import re

# folder_path = "data/text"
# txt_files = glob.glob(os.path.join(folder_path, "data1_*.txt"))

# # Sắp theo ID số trong tên file: data1_0001_..., data1_0002_..., ...
# txt_files = sorted(
#     txt_files,
#     key=lambda p: int(re.search(r"data1_(\d+)_", os.path.basename(p)).group(1))
# )

# tweet_sample = []
# for file_path in txt_files:
#     with open(file_path, "r", encoding="utf-8") as f:
#         tweet_sample.append(f.read())

# print(f"Đã đọc {len(tweet_sample)} file từ {folder_path}")


In [23]:
import re
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from textblob import Word
from nltk.util import ngrams
import re
from wordcloud import WordCloud, STOPWORDS
from nltk.tokenize import word_tokenize

In [24]:
# import re
# import unicodedata

# def deep_clean_text(text):
#     # 1. Chuẩn hóa Unicode NFC
#     text = unicodedata.normalize('NFC', text)

#     # 2. Xóa marker trang
#     text = re.sub(r'===\s*TRANG\s*\d+\s*===', ' ', text, flags=re.IGNORECASE)

#     # 3. Xóa chuỗi rác đặc trưng từ OCR/footer của ThuVienPhapLuat & LawSoft
#     noise_patterns = [
#         r'www\.ThuVienPhapLuat\.\w+',
#         r'ThuVienPhapLuat',
#         r'Sofi',
#         r'Jel',
#         r'Í\s*,?\s*aw\s*Soft\s*:?',
#         r'LawSoft\s*:?',
#         r'Jel\s*:\s*[\d\-]+',
#         r'lel\s*:\s*[\d\-]+',
#         r'Tel\s*:\s*[\d\-]+',
#         r'\d{4}\s+\d{4}',          # chuỗi số kiểu "3930 3279"
#         r'84[-\s]?\d+',            # mã quốc gia lẫn vào
#         r'\b[Vv][- ]?[Jj]\d*\s+r\s+-cj\s+[\wìãỉ]+\b',
#     ]
#     for pattern in noise_patterns:
#         text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

#     # 4. Xóa ký tự Latin lỗi font (ö, ø, ì, ã đứng độc lập không thuộc từ Việt)
#     text = re.sub(r'[öøæœ]', ' ', text)
#     text = re.sub(r'Xu\s+söôø\.?', ' ', text, flags=re.IGNORECASE)

#     # 5. Loại bỏ gạch chéo ngược
#     text = re.sub(r'\\', '', text)

#     # 6. Xóa số đứng độc lập (số trang, artifact OCR) — số 1-3 chữ số không nằm trong từ/cụm pháp lý
#     # Giữ lại số trong cụm như "Điều 3", "khoản 1", "năm 2020", "03 tháng"
#     text = re.sub(r'(?<![a-zA-ZÀ-ỹ\d])\d{1,3}(?![a-zA-ZÀ-ỹ\d\/\-])', ' ', text)

#     # 7. Whitelist — chỉ giữ ký tự tiếng Việt, số, dấu câu cơ bản
#     VIET_CHARS = (
#         'A-Za-z0-9'
#         'ÁÀẢÃẠÂẤẦẨẪẬĂẮẰẲẴẶ'
#         'ÉÈẺẼẸÊẾỀỂỄỆ'
#         'ÍÌỈĨỊ'
#         'ÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢ'
#         'ÚÙỦŨỤƯỨỪỬỮỰ'
#         'ÝỲỶỸỴ'
#         'áàảãạâấầẩẫậăắằẳẵặ'
#         'éèẻẽẹêếềểễệ'
#         'íìỉĩị'
#         'óòỏõọôốồổỗộơớờởỡợ'
#         'úùủũụưứừửữự'
#         'ýỳỷỹỵ'
#         'đĐ'
#         r'\s,.\-:/()\[\]'
#     )
#     text = re.sub(f'[^{VIET_CHARS}]', ' ', text)

#     # 8. Xóa các từ/cụm chỉ toàn chữ Latin không dấu đứng lẻ (artifact OCR như "VH", "CP", OK giữ lại vì là viết tắt pháp lý)
#     # Loại bỏ chữ Latin lẻ 1-2 ký tự không phải viết tắt quen thuộc
#     # (Tuỳ chỉnh nếu cần giữ lại các từ viết tắt cụ thể)
#     text = re.sub(r'(?<!\w)[A-Z]{1}(?!\w)', ' ', text)   # chữ hoa đứng lẻ
#     text = re.sub(r'(?<!\w)[a-z]{1,2}(?!\w)', ' ', text) # chữ thường 1-2 ký tự đứng lẻ

#     # 9. Xử lý dòng chỉ có số hoặc rỗng
#     lines = text.split('\n')
#     clean_lines = [
#         line for line in lines
#         if not re.match(r'^\s*[\d\s\W]{0,10}\s*$', line)
#     ]
#     text = ' '.join(clean_lines)

#     # 10. Hậu xử lý khoảng trắng
#     text = re.sub(r'\s+', ' ', text)
#     text = re.sub(r'\s+([,.\-:!?;])', r'\1', text)
#     text = re.sub(r'([,.!?;:]){2,}', r'\1', text)
#     text = re.sub(r'\.{2,}', '.', text)

#     return text.strip()

# tweet_sample = [deep_clean_text(tweet) for tweet in tweet_sample]

# print(tweet_sample[0])

In [25]:
# import re

# # Từ điển của bạn
# abbreviation_dict = {
#     "nđ": "nghị định",
#     "cp": "chính phủ",
#     "qđ": "quyết định",
#     "nq": "nghị quyết",
#     "ttg": "thủ tướng", 
#     "qđ-ttg": "quyết định thủ tướng chính phủ",
#     "tp": "thành phố",
#     "vn": "việt nam",
#     "ttr": "tờ trình", 
#     "kh": "kế hoạch",  
#     "ubnd": "ủy ban nhân dân",
#     "hđnd": "hội đồng nhân dân",
#     "qyđ": "quy định",
#     "vpcp": "văn phòng chính phủ",
#     "btcn": "bộ trưởng chủ nhiệm",
#     "pcn": "phó chủ nhiệm",
#     "ttgt": "thủ tướng chính phủ",
#     "tgđ": "tổng giám đốc",
#     "ttđt": "thông tin điện tử",
#     "vcl": "vị trí việc làm",
#     "tccv": "tổ chức cán bộ",
#     "ktvbqppl": "kiểm tra văn bản quy phạm pháp luật",
#     "gdđt": "giáo dục và đào tạo",
#     "nxb": "nhà xuất bản",
#     "qppl": "quy phạm pháp luật",
#     "hđ": "hội đồng"
# }

# # 1. Sắp xếp độ dài giảm dần để tránh khớp nhầm (như 'tt' trong 'ttg')
# sorted_abbrs = sorted(abbreviation_dict.keys(), key=len, reverse=True)

# # 2. QUAN TRỌNG: Thêm re.IGNORECASE để khớp cả chữ HOA và chữ thường
# pattern = re.compile(r'\b(' + '|'.join(map(re.escape, sorted_abbrs)) + r')\b', re.IGNORECASE)

# def expand_abbreviations(text):
#     def replace(match):
#         # Lấy từ khớp được và đưa về viết thường để tra cứu trong dict
#         found_word = match.group(0).lower()
#         return abbreviation_dict[found_word]
    
#     return pattern.sub(replace, text)

# # Chạy thử nghiệm trên tweet_sample
# for i in range(len(tweet_sample)):
#     tweet_sample[i] = expand_abbreviations(tweet_sample[i])

# print(tweet_sample[0])

In [26]:
pip install transformers -U

  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.44.2━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

In [27]:
pip install sentencepiece protobuf


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
# import torch
# import os
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# # Cấu hình môi trường
# os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# device = 0 if torch.cuda.is_available() else -1

# model_name = "bmd1905/vietnamese-correction-v2"

# # 1. Load model và tokenizer đúng chuẩn BARTpho
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

# # 2. Khởi tạo pipeline với task text2text-generation
# # Dùng model object trực tiếp để pipeline tự nhận diện cấu trúc Seq2Seq
# corrector = pipeline(
#     task="text-generation", 
#     model=model, 
#     tokenizer=tokenizer, 
#     device=device
# )

# def smart_chunk_text(text, max_words=60): 
#     words = text.split()
#     return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

# # Chạy thử nghiệm cho tệp Nghị định 69/2017/Nghị định-Chính phủ
# index_to_test = 1
# original_text = tweet_sample[index_to_test]
# text_chunks = smart_chunk_text(original_text)

# if text_chunks:
#     print(f"--- Đang xử lý file index {index_to_test} với {len(text_chunks)} chunks ---")
#     corrected_chunks = [] # Đảm bảo biến này đã được khởi tạo
    
#     try:
#         results = corrector(
#             text_chunks, 
#             batch_size=2, 
#             max_length=256 
#         )
        
#         # Xử lý kết quả an toàn cho cả dạng list of dict và list of list of dict
#         for res in results:
#             if isinstance(res, list): # Trường hợp results là list lồng nhau
#                 corrected_chunks.append(res[0]['generated_text'])
#             elif isinstance(res, dict): # Trường hợp results là list các dict
#                 corrected_chunks.append(res['generated_text'])
            
#         corrected_text = ' '.join(corrected_chunks)
        
#         # Hiển thị kết quả so sánh
#         print("\n[BẢN GỐC]:", original_text)
#         print("\n" + "="*50)
#         print("[SAU SỬA LỖI]:", corrected_text)
                   
#     except Exception as e:
#         print(f"❌ Lỗi khi xử lý: {e}")

In [29]:
# import os
# import re

# output_folder = "data/cleaned"
# os.makedirs(output_folder, exist_ok=True)

# def save_cleaned_files(original_paths, cleaned_contents):
#     for file_path, content in zip(original_paths, cleaned_contents):
#         filename = os.path.basename(file_path)
        
#         save_path = os.path.join(output_folder, filename)
        
#         try:
#             with open(save_path, 'w', encoding='utf-8') as f:
#                 f.write(content)
#         except Exception as e:
#             print(f"Lỗi khi lưu file {filename}: {e}")

#     print(f"✅ Đã lưu xong {len(cleaned_contents)} file vào {output_folder}")

# # Thực hiện lưu
# save_cleaned_files(txt_files, tweet_sample)

In [30]:
pip install langchain openai tiktoken langchain-text-splitters langchain-experimental langchain-community sentence-transformers tf-keras 


  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.13.0
    Uninstalling huggingface_hub-1.13.0:
      Successfully uninstalled huggingface_hub-1.13.0
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [huggingface-hub]
    Found existing installation: transformers 5.7.032m0/2 [huggingface-hub]
    Uninstalling transformers-5.7.0:m╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-5.7.0━━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use upd

In [31]:
pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
!pip install safetensors


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import torch
print(torch.__version__)

2.5.1+cu121


In [34]:
pip install -U langchain-huggingface


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [35]:
pip install transformers==4.46.3 pandas sentence-transformers

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached tokenizers-0.20.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.46.3-py3-none-any.whl (10.0 MB)
Using cached tokenizers-0.20.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.57.6━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updat

In [36]:
# Chạy lệnh này trong 1 cell mới
!pip uninstall transformers sentence-transformers tokenizers -y
!pip install transformers==4.44.2 sentence-transformers==3.0.1 tokenizers==0.19.1

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: sentence-transformers 3.0.1
Uninstalling sentence-transformers-3.0.1:
  Successfully uninstalled sentence-transformers-3.0.1
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl.metadata (10 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sentence-transformers]ence-transformers]

[notice] A new relea

In [ ]:
import pandas as pd
import re
import os
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "BAAI/bge-m3"

# Chỉ tạo 1 lần trong kernel hiện tại
if "embeddings" not in globals():
    print("Dang khoi tao embeddings 1 lan...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL_NAME,
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )
else:
    print("Da co embeddings, tai su dung.")

# 1. Chuẩn bị Metadata Dictionary từ CSV
def load_metadata_lookup(csv_path):
    df = pd.read_csv(csv_path)
    # zfill(4) để đảm bảo ID "456" khớp với "0456" trong tên file
    df['ID'] = df['ID'].astype(str).str.zfill(4) 
    return df.set_index('ID').to_dict('index')

# 2. Hàm xử lý chính
def process_txt_files_to_chunks(file_paths, csv_path, embeddings):
    # Load metadata lookup table
    metadata_lookup = load_metadata_lookup(csv_path)
    
    splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile", 
        breakpoint_threshold_amount=95
    )

    all_chunks = []

    for file_path in tqdm(file_paths, desc="Đang xử lý file"):
        filename = os.path.basename(file_path)
        
        # Trích xuất ID từ tên file (ví dụ: data1_0456_...)
        match = re.search(r'data1_(\d+)_', filename)
        if not match:
            continue
            
        file_id = match.group(1)
        meta_info = metadata_lookup.get(file_id, {})

        # Đọc nội dung thực tế từ file .txt
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Thực hiện Semantic Chunking trên nội dung file
            chunks = splitter.split_text(content)
            
            for j, chunk in enumerate(chunks):
                if len(chunk.strip()) < 50: continue # Bỏ qua chunk rác
                
                all_chunks.append({
                    "chunk_id": f"id{file_id}_c{j}",
                    "text": chunk,
                    "metadata": {
                        "id": file_id,
                        "so_hieu": meta_info.get('So_Hieu', 'N/A'),
                        "ngay_ban_hanh": meta_info.get('Ngay_Ban_Hanh', 'N/A'),
                        "linh_vuc": meta_info.get('Linh_Vuc', 'N/A'),
                        "co_quan": meta_info.get('Co_Quan_Ban_Hanh', 'N/A'),
                        "link": meta_info.get('Link_Download', ''),
                        "file_source": filename
                    }
                })
        except Exception as e:
            print(f"Lỗi khi xử lý file {filename}: {e}")

    return all_chunks


cleaned_files = [os.path.join("data/cleaned", f) for f in os.listdir("data/cleaned") if f.endswith('.txt')]

all_chunks = process_txt_files_to_chunks(
    cleaned_files,
    "danh_sach_van_ban_dut.csv",
    embeddings
)
print(f"Đã tạo {len(all_chunks)} chunks từ {len(os.listdir('data/cleaned'))} documents")
print(all_chunks[0])

Dang khoi tao embeddings 1 lan...


Đang xử lý file:   0%|          | 1/389 [00:05<35:27,  5.48s/it]

In [ ]:
import json
import os

# Thư mục lưu chunks
chunk_output_folder = "data/chunks"
os.makedirs(chunk_output_folder, exist_ok=True)

# Đường dẫn file JSON
json_path = os.path.join(chunk_output_folder, "all_semantic_chunks.json")

# Lưu list all_chunks vào file JSON
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=4)

print(f"✅ Đã lưu thành công {len(all_chunks)} chunks vào file: {json_path}")

✅ Đã lưu thành công 12798 chunks vào file: data/chunks/all_semantic_chunks.json


In [ ]:
pip install --upgrade huggingface_hub datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.44.2 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.11.0 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.11.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


: 

In [ ]:
# ⭐ OPTIMIZED: Xử lý chunk by chunk + checkpoint để tránh OOM
import time
import gc
import json
import os
import torch
from tqdm.auto import tqdm
from langchain_huggingface import HuggingFaceEmbeddings

# 1) Load embeddings nếu chưa có
if "embeddings" not in globals():
    print("📥 Đang tải model BAAI/bge-m3...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
else:
    print("✅ Tái sử dụng embeddings đã có")

# 2) Checkpoint file để lưu tiến độ
checkpoint_file = "data/chunks/all_chunks_with_vectors.json"
os.makedirs("data/chunks", exist_ok=True)

# 3) Kiểm tra và load checkpoint nếu có
if os.path.exists(checkpoint_file):
    print("📂 Đang load checkpoint...")
    with open(checkpoint_file, 'r', encoding='utf-8') as f:
        all_chunks_processed = json.load(f)
    start_idx = len([c for c in all_chunks_processed if "vector" in c])
    print(f"✅ Đã load {start_idx}/{len(all_chunks_processed)} chunks với vectors")
else:
    all_chunks_processed = all_chunks.copy()
    start_idx = 0
    print(f"🆕 Bắt đầu xử lý từ đầu: {len(all_chunks_processed)} chunks")

# 4) ⭐ LOOP CHÍNH: Xử lý từng chunk một, KHÔNG tạo list lớn
print(f"\n⚙️ Bắt đầu mã hóa từ chunk {start_idx}...")
start_time = time.time()
use_cpu = False

for i in tqdm(range(start_idx, len(all_chunks_processed)), desc="Embedding"):
    chunk = all_chunks_processed[i]
    
    # Bỏ qua nếu đã có vector
    if "vector" in chunk:
        continue
    
    text = chunk["text"]
    
    try:
        # ⭐ KEY: Xử lý 1 chunk tại một thời điểm
        if use_cpu:
            vectors = embeddings_cpu.embed_documents([text])
        else:
            vectors = embeddings.embed_documents([text])
        
        chunk["vector"] = vectors[0]
        
        # Dọn cache sau mỗi chunk
        gc.collect()
        if torch.cuda.is_available() and not use_cpu:
            torch.cuda.empty_cache()
    
    except torch.OutOfMemoryError:
        if not use_cpu:
            print(f"\n⚠️ OOM tại chunk {i}, chuyển sang CPU...")
            use_cpu = True
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            embeddings_cpu = HuggingFaceEmbeddings(
                model_name="BAAI/bge-m3",
                model_kwargs={"device": "cpu"},
                encode_kwargs={"normalize_embeddings": True}
            )
            # Thử lại chunk hiện tại trên CPU
            vectors = embeddings_cpu.embed_documents([text])
            chunk["vector"] = vectors[0]
        else:
            print(f"\n❌ OOM ngay cả trên CPU! Dừng tại chunk {i}")
            break
    
    # 🔄 Lưu checkpoint mỗi 20 chunks để an toàn
    if (i + 1) % 20 == 0:
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(all_chunks_processed, f, ensure_ascii=False, indent=2)

end_time = time.time()

# 5) Kiểm tra kết quả
vectors_count = len([c for c in all_chunks_processed if "vector" in c])
print(f"\n✅ Hoàn tất!")
print(f"⏱️ Thời gian chạy: {end_time - start_time:.2f} giây")
print(f"📊 Đã mã hóa: {vectors_count}/{len(all_chunks_processed)} chunks")

if vectors_count == len(all_chunks_processed):
    print("🎉 Tất cả chunks đã có vectors!")
    # Update all_chunks để khớp với processed version
    all_chunks = all_chunks_processed
else:
    print(f"⚠️ Chưa hoàn tất, còn {len(all_chunks_processed) - vectors_count} chunks")

# 6) Lưu final checkpoint
with open(checkpoint_file, 'w', encoding='utf-8') as f:
    json.dump(all_chunks_processed, f, ensure_ascii=False, indent=2)
print(f"💾 Đã lưu checkpoint tại: {checkpoint_file}")

✅ Tái sử dụng embeddings đã có
⚙️ Bắt đầu mã hóa 12798 chunks (batch_size=1)...


Embedding:   0%|          | 0/12798 [00:00<?, ?it/s]

Embedding:  25%|██▍       | 3148/12798 [02:17<07:00, 22.92it/s]


⚠️ OOM trên GPU, chuyển sang CPU để hoàn tất mã hóa...


Embedding-CPU:  25%|██▍       | 393/1600 [39:16<1:59:28,  5.94s/it]

In [ ]:
# 📊 Kiểm tra status embedding
import json
checkpoint_file = "data/chunks/all_chunks_with_vectors.json"

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    
    with_vec = len([c for c in loaded if "vector" in c])
    total = len(loaded)
    pct = (with_vec / total * 100) if total > 0 else 0
    
    print(f"📈 Status: {with_vec}/{total} chunks ({pct:.1f}%)")
    
    if with_vec == total:
        print("✅ Hoàn tất! Sẵn sàng tải lên Qdrant")
    else:
        print(f"⏳ Còn {total - with_vec} chunks cần xử lý - chạy cell Embedding lần nữa")
else:
    print("❌ Chưa có checkpoint - chạy cell Embedding trước")

In [ ]:
print(all_chunks[0]["vector"])

In [ ]:
pip install qdrant-client

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import json

# 1. Load data từ checkpoint (hoặc từ all_chunks nếu chạy trực tiếp)
checkpoint_file = "data/chunks/all_chunks_with_vectors.json"

if os.path.exists(checkpoint_file):
    print("📂 Đang load chunks từ checkpoint...")
    with open(checkpoint_file, 'r', encoding='utf-8') as f:
        chunks_data = json.load(f)
else:
    print("📦 Sử dụng all_chunks từ kernel")
    chunks_data = all_chunks

# Kiểm tra xem tất cả chunks đã có vectors chưa
chunks_with_vectors = [c for c in chunks_data if "vector" in c]
print(f"✅ Sẽ tải {len(chunks_with_vectors)}/{len(chunks_data)} chunks")

if len(chunks_with_vectors) == 0:
    print("❌ Không có chunks nào có vectors! Chạy cell Embedding trước.")
else:
    # 2. Khởi tạo Qdrant Client
    client = QdrantClient(path="./qdrant_db")
    collection_name = "chatbot_documents"

    # 3. Tạo Collection
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
        )
        print(f"📦 Đã tạo mới Collection: {collection_name}")
    else:
        print(f"📦 Collection '{collection_name}' đã tồn tại.")

    # 4. Chuẩn bị Points
    points = []
    for i, chunk in enumerate(chunks_with_vectors):
        if "vector" not in chunk:
            continue
            
        payload = {
            "text": chunk["text"],
            "chunk_id": chunk.get("chunk_id", f"chunk_{i}"),
            "metadata": chunk.get("metadata", {})
        }
        
        point = PointStruct(
            id=i, 
            vector=chunk["vector"], 
            payload=payload
        )
        points.append(point)

    # 5. Upsert dữ liệu
    print(f"⚙️ Đang tải {len(points)} khối dữ liệu lên Qdrant Database...")
    client.upsert(
        collection_name=collection_name,
        points=points
    )

    print("✅ Hoàn tất! Toàn bộ Vector đã được tải lên Qdrant.")

In [ ]:
collection_name = "chatbot_documents"

result = client.retrieve(
    collection_name=collection_name,
    ids=[0], 
    with_payload=True,   # True để in ra phần text (Payload)
    with_vectors=False   # Tạm thời False để không in ra 1024 con số làm rối màn hình
)

# In kết quả
if result:
    print("✅ Dữ liệu Text đã lưu:")
    print(result[0].payload)
else:
    print("Không tìm thấy dữ liệu.")

In [ ]:
!pip install matplotlib seaborn wordcloud

In [ ]:
#9. analysis 

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import numpy as np

plt.rcParams['figure.figsize'] = (12, 6)

all_words = []
doc_lengths = []

for text in tweet_sample:
    words = text.split() 
    all_words.extend(words)
    doc_lengths.append(len(words))

print("-" * 30)
print(f"1. Tổng số văn bản (Documents): {len(tweet_sample)}")
print(f"2. Tổng số từ (Total Words): {len(all_words)}")
print(f"3. Số từ vựng duy nhất (Vocabulary Size): {len(set(all_words))}")
print(f"4. Độ dài trung bình văn bản: {np.mean(doc_lengths):.2f} từ")
print("-" * 30)


text_for_cloud = " ".join(tweet_sample)


font_path = "C:/Windows/Fonts/arial.ttf" 

try:
    wordcloud = WordCloud(
        width=1600, height=800,
        background_color='white',
        font_path=font_path, # Quan trọng để hiển thị tiếng Việt
        colormap='viridis',
        max_words=200
    ).generate(text_for_cloud)

    plt.figure(figsize=(15, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"⚠️ Lỗi vẽ WordCloud (có thể do font): {e}")


word_counts = Counter(all_words)
most_common_words = word_counts.most_common(20)

words = [w[0] for w in most_common_words]
counts = [w[1] for w in most_common_words]

plt.figure(figsize=(12, 6))
sns.barplot(x=counts, y=words, palette='viridis')
plt.title('Top 20 từ xuất hiện nhiều nhất', fontsize=15)
plt.xlabel('Số lần xuất hiện')
plt.ylabel('Từ')
plt.show()


word_lengths = [len(w) for w in all_words]

plt.figure(figsize=(12, 6))
sns.histplot(word_lengths, bins=20, kde=True, color='orange')
plt.title('Phân bố độ dài của từ (Số ký tự)', fontsize=15)
plt.xlabel('Độ dài từ')
plt.ylabel('Tần suất')
plt.show()

